In [ ]:
import torch

In [ ]:
inputs=torch.tensor([
    [0.43,0.15,0.89], # your
    [0.55,0.87,0.66], # journey
    [0.57,0.85,0.64], # starts
    [0.22,0.58,0.23], # with
    [0.77,0.25,0.10], # one
    [0.05,0.80,0.55]  # step
])

In [ ]:
batch=torch.stack((inputs,inputs),dim=0)
print(batch.shape)
d_in,d_out=3,3

In [ ]:
class CausalAttention(torch.nn.Module):
    def __init__(self, d_in, d_out,context_length,drop_out):
        
        super().__init__()
        self.query=torch.nn.Linear(d_in, d_out, bias=False)
        self.key=torch.nn.Linear(d_in, d_out, bias=False)
        self.value=torch.nn.Linear(d_in, d_out, bias=False)
        self.d_out=d_out
        self.d_in=d_in
        self.context_length=context_length
        self.drop_out=torch.nn.Dropout(drop_out)
        self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1)) # not trained model to ensure not happening of device mis match error
    
    def forward(self,x):
        b,num_tokens,d_in=x.shape
        Query=self.query(x)
        Key=self.key(x)
        Value=self.value(x)
        
        attn_score= Query @ Key.transpose(1,2)
        attn_score.masked_fill_(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)
        attn_weights=torch.softmax(attn_score/Key.shape[-1]**0.5,dim=-1)
        attn_weights= self.drop_out(attn_weights)
        context_vector= attn_weights @ Value
        return context_vector
    

In [ ]:
torch.manual_seed(123)
context_length=batch.shape[1]
ca=CausalAttention(d_in,d_out,context_length,0.3)
context=ca(batch)

In [ ]:
print(context.shape)